<a href="https://colab.research.google.com/github/OscarPellicer/extra-attributions/blob/main/notebooks/02_dl_attributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 · Hands-on: explaining a CNN

We explain the predictions of an ImageNet CNN (**RegNetY-3.2GF**) with [Captum](https://captum.ai): gradient-based methods, Integrated Gradients, SmoothGrad (`NoiseTunnel`), Grad-CAM, a **contrastive** explanation (*why hen and not rooster?*) and a **sanity check** (Adebayo et al., 2018).

The image contains a hen **and** a rooster, so every method is shown for the **two top classes** (*cock* and *hen*): how much does each map change with the class?

Run the cells in order. Parameters marked `# change me` list values worth trying: change one, re-run the cell and compare. The first run downloads the model weights (~75 MB).

*A longer version (VGG16, LRP, Quantus metrics, with exercises and answers) is in `extended/02_dl_attributions.ipynb`.*

In [ ]:
# Install the dependencies (only on Google Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q captum quantus lovely-tensors git+https://github.com/OscarPellicer/extra-attributions.git
else:
    print('Running locally. If anything is missing, from the root of the extra-attributions repo run:\n'
          '    pip install -r requirements.txt\n    pip install -e .')

In [ ]:
import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')  # Apple-silicon: fall back to CPU for unsupported ops

import copy, json, time, urllib.request, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

from captum.attr import (Saliency, GuidedBackprop, InputXGradient, IntegratedGradients, NoiseTunnel,
                         LayerGradCam, GuidedGradCam, LayerAttribution, Occlusion, LRP)
from captum.attr._utils.lrp_rules import EpsilonRule, GammaRule

import lovely_tensors as lt
lt.monkey_patch()  # nicer printing of tensors
warnings.filterwarnings('ignore', message='.*required_grads has been set automatically.*')
warnings.filterwarnings('ignore', message='.*Setting backward hooks on ReLU activations.*')

# Device: CUDA > Apple MPS > CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f'Using {DEVICE=} | torch {torch.__version__}')
torch.manual_seed(0); np.random.seed(0)

Helpers: image loading and plotting. **Green** = evidence *for* the class, **red** = *against*.

In [ ]:
MEDIA_URL = 'https://raw.githubusercontent.com/OscarPellicer/extra-attributions/main/notebooks/media/'

def get_media(fname):
    '''Return a local path to `fname`, downloading it into ./media if needed.'''
    for folder in [Path('media'), Path('../media'), Path('notebooks/media')]:
        if (folder / fname).exists():
            return folder / fname
    Path('media').mkdir(exist_ok=True)
    urllib.request.urlretrieve(MEDIA_URL + fname, Path('media') / fname)
    return Path('media') / fname

# Red-white-green colour map for signed attributions
_colors, _values = ['darkred', 'red', 'lightcoral', 'white', 'palegreen', 'green', 'darkgreen'], [0, .15, .4, .5, .6, .9, 1.]
RdGn = LinearSegmentedColormap.from_list('RdGn', list(zip(_values, _colors)), N=256)

def to_2d(attr):
    '''(1,C,H,W) / (C,H,W) / (H,W) attribution -> (H,W) numpy array (mean over channels).'''
    a = torch.as_tensor(attr).detach().float().cpu()
    while a.dim() > 3:
        a = a[0]
    return (a.mean(0) if a.dim() == 3 else a).numpy()

def to_rgb(image):
    return image.detach().cpu().permute(1, 2, 0).clamp(0, 1).numpy()

def plot_attributions(image, attributions=[], attribution_names=[], limits=None, title=None,
                      figsize=3, percentile=99.5):
    '''Plot the input image followed by one signed attribution map per method.'''
    n = 1 + len(attributions)
    fig, axes = plt.subplots(1, n, figsize=(figsize * n, figsize + 0.5), squeeze=False)
    axes = axes[0]
    axes[0].imshow(to_rgb(image)); axes[0].set_title('Input')
    for i, (ax, attr, name) in enumerate(zip(axes[1:], attributions, attribution_names)):
        m = to_2d(attr)
        lim = limits[i] if isinstance(limits, list) else limits
        if lim is None:
            v = max(np.percentile(np.abs(m), percentile), 1e-12)
            lim = (-v, v)
        im = ax.imshow(m, cmap=RdGn, vmin=lim[0], vmax=lim[1])
        ax.set_title(name)
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03); cb.ax.tick_params(labelsize=7)
    for ax in axes:
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout(); plt.show()

def plot_overlay(image, heatmaps, names, title=None, figsize=3):
    '''Classic Grad-CAM style: non-negative heatmaps overlaid on the image.'''
    n = 1 + len(heatmaps)
    fig, axes = plt.subplots(1, n, figsize=(figsize * n, figsize + 0.5), squeeze=False)
    axes = axes[0]
    axes[0].imshow(to_rgb(image)); axes[0].set_title('Input')
    for ax, h, name in zip(axes[1:], heatmaps, names):
        h = to_2d(h)
        ax.imshow(to_rgb(image)); ax.imshow(h / (h.max() + 1e-12), cmap='jet', alpha=0.5, vmin=0, vmax=1)
        ax.set_title(name)
    for ax in axes:
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout(); plt.show()

In [ ]:
# RegNetY-3.2GF (Radosavovic et al., 2020): ~82% top-1 on ImageNet, 19M parameters (~75 MB download).
# Every ReLU is a separate module (unlike ResNets), which Guided Backprop needs.
model = models.regnet_y_3_2gf(weights=models.RegNet_Y_3_2GF_Weights.IMAGENET1K_V2).eval().to(DEVICE)
for m in model.modules():
    if isinstance(m, nn.ReLU):
        m.inplace = False   # in-place ReLUs break the backward hooks used by Guided Backprop

with open(get_media('imagenet_class_index.json')) as f:
    idx_to_labels = json.load(f)
def idx_to_label(i):
    return idx_to_labels[str(int(i))][1]
label_to_idx = {v[1]: int(k) for k, v in idx_to_labels.items()}

# We resize the whole image to 224x224 (instead of a center crop) so that nothing is cut out
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def load_image(fname, crop=None):
    '''Returns (image in [0,1] (3,224,224) on CPU, normalised input batch (1,3,224,224) on DEVICE).'''
    img = Image.open(get_media(fname)).convert('RGB')
    if crop is not None:
        img = img.crop(crop)
    x_img = transform(img)
    return x_img, normalize(x_img)[None].to(DEVICE)

def show_top5(x):
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)
    top_p, top_i = probs.topk(5)
    for k, (p, i) in enumerate(zip(top_p[0], top_i[0])):
        print(f' {k+1}. {idx_to_label(i):<20s} {p.item()*100:6.2f}%   (class index {i.item()})')
    return top_i[0].tolist()

In [ ]:
IMAGE = 'hen_rooster.jpg'   # change me: 'hen_rooster.jpg' or 'swan.jpg'

x_img, x = load_image(IMAGE)
print(x)  # lovely_tensors summary of the normalised input
top5 = show_top5(x)
TARGETS = top5[:2]          # change me: classes to explain, e.g. [label_to_idx['hen'], label_to_idx['goldfish']]
print('Explaining:', [idx_to_label(t) for t in TARGETS])
plt.figure(figsize=(3, 3)); plt.imshow(to_rgb(x_img)); plt.axis('off'); plt.title(IMAGE); plt.show()

## 1. Gradient-based methods

* **Saliency**: $\partial f_c / \partial x$, which pixels change the class score most.
* **Guided Backprop**: at every ReLU, also zero the negative gradients. Very crisp maps... but do they change with the class?
* **Input×Gradient**: $x \odot \partial f_c / \partial x$.

In [ ]:
for t in TARGETS:
    attr_sal = Saliency(model).attribute(x, target=t, abs=False)
    attr_gbp = GuidedBackprop(model).attribute(x, target=t)
    attr_ixg = InputXGradient(model).attribute(x, target=t)
    plot_attributions(x_img, [attr_sal, attr_gbp, attr_ixg], ['Saliency', 'Guided Backprop', 'Input×Gradient'],
                      title=f'Target: {idx_to_label(t)}')

## 2. Integrated Gradients

IG integrates the gradient along a path from a **baseline** $x'$ to the input, and its attributions add up to $f_c(x) - f_c(x')$ (`delta` = error of the numerical integral: if it is large, the map is not reliable). RegNet is quite non-linear along this path, so it needs many steps (VGG16 was fine with ~32). The baseline means "absence of signal": with a **black** baseline, black pixels (the rooster's body, the background) always get zero attribution.

In [ ]:
def make_baseline(kind, x_img):
    '''Baselines are built in [0,1] image space and then normalised like the input.'''
    if kind == 'black':
        b = torch.zeros_like(x_img)
    elif kind == 'white':
        b = torch.ones_like(x_img)
    elif kind == 'mean':      # = 0 in normalised space
        b = torch.tensor([0.485, 0.456, 0.406])[:, None, None].expand_as(x_img)
    elif kind == 'blur':
        b = transforms.GaussianBlur(kernel_size=51, sigma=20)(x_img)
    elif kind == 'uniform noise':
        b = torch.rand(x_img.shape, generator=torch.Generator().manual_seed(0))
    else:
        raise ValueError(kind)
    return normalize(b)[None].to(DEVICE)

In [ ]:
N_STEPS = 256                                            # change me: e.g. 8, 32, 128, 256, 512 (watch delta)
BASELINES = ['black', 'mean', 'blur', 'uniform noise']   # change me: also 'white'

ig = IntegratedGradients(model)
for t in TARGETS:
    attrs = []
    for kind in BASELINES:
        baseline = make_baseline(kind, x_img)
        attr, delta = ig.attribute(x, baselines=baseline, target=t, n_steps=N_STEPS,
                                   internal_batch_size=16, return_convergence_delta=True)
        with torch.no_grad():
            diff = (model(x)[0, t] - model(baseline)[0, t]).item()
        print(f'{idx_to_label(t):>6s} | {kind:>14s} baseline | sum(IG) = {attr.sum().item():7.3f} '
              f'| f(x) - f(baseline) = {diff:7.3f} | delta = {delta.item():+.4f}')
        attrs.append(attr)
    plot_attributions(x_img, attrs, [f'IG ({k} baseline)' for k in BASELINES],
                      title=f'Target: {idx_to_label(t)}  |  n_steps = {N_STEPS}')

## 3. SmoothGrad (`NoiseTunnel`)

Gradients of deep networks fluctuate sharply at small scales. **SmoothGrad** averages the attribution over `NT_SAMPLES` noisy copies of the input. In Captum, `NoiseTunnel` wraps *any* method: here Saliency (absolute values, as in the SmoothGrad paper) and Input×Gradient, with and without noise.

In [ ]:
NOISE_LEVEL = 0.15   # change me: e.g. 0.0, 0.05, 0.15, 0.5 (noise std as a fraction of the input range)
NT_SAMPLES = 16      # change me: e.g. 4, 16, 64

nt_kwargs = dict(nt_type='smoothgrad', nt_samples=NT_SAMPLES, nt_samples_batch_size=8,
                 stdevs=NOISE_LEVEL * (x.max() - x.min()).item())
for t in TARGETS:
    attr_sal = Saliency(model).attribute(x, target=t, abs=True)
    attr_sal_nt = NoiseTunnel(Saliency(model)).attribute(x, target=t, abs=True, **nt_kwargs)
    attr_ixg = InputXGradient(model).attribute(x, target=t)
    attr_ixg_nt = NoiseTunnel(InputXGradient(model)).attribute(x, target=t, **nt_kwargs)
    plot_attributions(x_img, [attr_sal, attr_sal_nt, attr_ixg, attr_ixg_nt], ['|Saliency|', '|Saliency| + NT', 'Input×Gradient', 'IxG + NT'],
                      title=f'Target: {idx_to_label(t)}  |  noise = {NOISE_LEVEL:.0%}, samples = {NT_SAMPLES}')

## 4. Grad-CAM

Grad-CAM weights the channels of a convolutional layer by their average gradient. The map has the resolution of that layer (7×7 for the last stage of RegNet, upsampled), and **Guided Grad-CAM** multiplies it by Guided Backprop. Is it really class-discriminative?

In [ ]:
GRADCAM_LAYER = 'trunk_output.block4'   # change me: 'trunk_output.block4' (7x7), 'trunk_output.block3' (14x14), 'trunk_output.block2' (28x28)

layer = model.get_submodule(GRADCAM_LAYER)
heatmaps = []
for t in TARGETS:
    attr_gc = LayerGradCam(model, layer).attribute(x, target=t, relu_attributions=True)
    attr_gc_up = LayerAttribution.interpolate(attr_gc, x.shape[-2:], interpolate_mode='bilinear')
    attr_ggc = GuidedGradCam(model, layer).attribute(x, target=t)
    heatmaps.append(attr_gc_up)
    plot_attributions(x_img, [attr_gc, attr_gc_up, attr_ggc],
                      [f'Grad-CAM (raw {attr_gc.shape[-2]}x{attr_gc.shape[-1]})', 'Grad-CAM (upsampled)', 'Guided Grad-CAM'],
                      title=f'Target: {idx_to_label(t)}')
plot_overlay(x_img, heatmaps, [f'Grad-CAM: {idx_to_label(t)}' for t in TARGETS])

## 5. Contrastive explanation: *why hen and not rooster?*

Explaining $f_\text{hen}(x)$ alone mixes "it is a chicken" (shared by both classes) with "it is a hen rather than a rooster". Explaining the **difference of logits** $f_A(x) - f_B(x)$ removes the shared evidence.

In [ ]:
class LogitDifference(nn.Module):
    '''Wraps a classifier so that its single output is logit[class_a] - logit[class_b].'''
    def __init__(self, model, class_a, class_b):
        super().__init__()
        self.model, self.class_a, self.class_b = model, class_a, class_b
    def forward(self, x):
        logits = self.model(x)
        return logits[:, [self.class_a]] - logits[:, [self.class_b]]

In [ ]:
CLASS_A = 'hen'                       # change me
CLASS_B = 'cock'                      # change me: e.g. 'cock', 'partridge', 'goldfish'
CONTRAST_IMAGE = 'hen_rooster.jpg'    # change me

xc_img, xc = load_image(CONTRAST_IMAGE)
a, b = label_to_idx[CLASS_A], label_to_idx[CLASS_B]
with torch.no_grad():
    logits = model(xc)[0]
print(f'logit[{CLASS_A}] = {logits[a]:.2f}, logit[{CLASS_B}] = {logits[b]:.2f}, difference = {logits[a] - logits[b]:.2f}')

models_to_explain = {CLASS_A: (model, a), CLASS_B: (model, b),
                     f'{CLASS_A} − {CLASS_B}': (LogitDifference(model, a, b), 0),
                     f'{CLASS_B} − {CLASS_A}': (LogitDifference(model, b, a), 0)}
occ_maps, gc_maps = {}, {}
for name, (f, t) in models_to_explain.items():
    gc = LayerGradCam(f, model.get_submodule(GRADCAM_LAYER)).attribute(xc, target=t, relu_attributions=True)
    gc_maps[name] = LayerAttribution.interpolate(gc, xc.shape[-2:], interpolate_mode='bilinear')
    occ_maps[name] = Occlusion(f).attribute(xc, target=t, sliding_window_shapes=(3, 32, 32), strides=(3, 16, 16),
                                            baselines=0, perturbations_per_eval=32)

names = list(models_to_explain)
plot_overlay(xc_img, [gc_maps[n] for n in names], [f'Grad-CAM: {n}' for n in names], title='Grad-CAM')
plot_attributions(xc_img, [occ_maps[n] for n in names[:3]], [f'Occlusion: {n}' for n in names[:3]],
                  title='Occlusion (w=32, s=16)')

## 6. Sanity check: randomise the model

If an explanation is faithful to the model, it must **change when the model is destroyed**. We re-initialise the network stage by stage, from the classifier (`fc`) down to the input (`stem`), and compare each map with the original one (Spearman correlation). Which method keeps showing the birds when the network is random?

In [ ]:
SANITY_METHODS = ['Saliency', 'Guided Backprop', 'Integrated Gradients', 'Grad-CAM']

def explain(method, model, x, target):
    """Returns an (N,C,H,W) attribution tensor."""
    if method == 'Saliency':
        return Saliency(model).attribute(x, target=target, abs=False)
    if method == 'Guided Backprop':
        return GuidedBackprop(model).attribute(x, target=target)
    if method == 'Integrated Gradients':
        return IntegratedGradients(model).attribute(x, target=target, n_steps=16, internal_batch_size=16)
    if method == 'Grad-CAM':
        gc = LayerGradCam(model, model.trunk_output.block4).attribute(x, target=target, relu_attributions=True)
        return LayerAttribution.interpolate(gc, x.shape[-2:], interpolate_mode='bilinear')
    raise ValueError(method)

# RegNet: stem -> 4 stages (block1 ... block4) -> classifier (fc). We randomise whole stages, from the output to the input
STAGES = ['fc', 'trunk_output.block4', 'trunk_output.block3', 'trunk_output.block2', 'trunk_output.block1', 'stem']

def randomise(module):
    for m in module.modules():
        if hasattr(m, 'reset_parameters'):
            m.reset_parameters()   # new random weights (and reset BatchNorm statistics)

In [ ]:
RANDOMISATION = 'cascading'   # change me: 'cascading' (top-down, cumulative) or 'independent' (one stage at a time)
ABS = True                    # change me: compare |attributions| (as in the paper) or signed attributions
TARGET = TARGETS[0]

def attribution_map(method, m):
    a = explain(method, m, x, TARGET).detach().sum(1)[0]   # sum over colour channels -> (H, W)
    return (a.abs() if ABS else a).cpu().numpy()

def similarity(a, b):
    if a.std() == 0 or b.std() == 0:   # e.g. an all-zero Grad-CAM map
        return np.nan
    return spearmanr(a.ravel(), b.ravel()).statistic

torch.manual_seed(0)
t0 = time.perf_counter()
stages = ['original'] + STAGES
maps = {meth: [attribution_map(meth, model)] for meth in SANITY_METHODS}
rand_model = copy.deepcopy(model)
for stage in STAGES:
    if RANDOMISATION == 'independent':
        rand_model = copy.deepcopy(model)            # start again from the trained model
    randomise(rand_model.get_submodule(stage))
    for meth in SANITY_METHODS:
        maps[meth].append(attribution_map(meth, rand_model))
scores = {meth: [similarity(maps[meth][0], m) for m in maps[meth]] for meth in SANITY_METHODS}
print(f'Done in {time.perf_counter() - t0:.1f}s (explaining {idx_to_label(TARGET)})')

# Maps at each randomisation stage (each map normalised to its own range)
short = lambda s: s.split('.')[-1]
fig, axes = plt.subplots(len(SANITY_METHODS), len(stages), figsize=(1.9 * len(stages), 1.9 * len(SANITY_METHODS)), squeeze=False)
for r, meth in enumerate(SANITY_METHODS):
    for c, s in enumerate(stages):
        m, ax = maps[meth][c], axes[r, c]
        if ABS:
            ax.imshow(m, cmap='gray', vmin=0, vmax=max(np.percentile(m, 99.5), 1e-12))
        else:
            v = max(np.percentile(np.abs(m), 99.5), 1e-12); ax.imshow(m, cmap=RdGn, vmin=-v, vmax=v)
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0: ax.set_title(s if s == 'original' else (f'→ {short(s)}' if RANDOMISATION == 'cascading' else f'only {short(s)}'), fontsize=9)
        if c == 0: ax.set_ylabel(meth, fontsize=9)
fig.suptitle(f'{RANDOMISATION.capitalize()} randomisation of RegNetY-3.2GF (from the output)')
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 3.5))
for meth in SANITY_METHODS:
    plt.plot(range(len(stages)), scores[meth], 'o-', label=meth)
plt.xticks(range(len(stages)), [short(s) for s in stages])
plt.axhline(0, color='grey', lw=0.8); plt.ylim(-0.3, 1.05)
plt.ylabel('Spearman ρ vs. original'); plt.xlabel(f'{RANDOMISATION} randomisation up to stage →')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 7. *(Optional)* Methods from `extra_attributions`

The package [`extra_attributions`](https://github.com/OscarPellicer/extra-attributions) (by the instructor) implements three methods with Captum's interface:

* `TiledOcclusion`: runs Occlusion several times on grids shifted by `window / K` and combines them: `K` times finer maps without shrinking the occlusion window.
* `FusionGrad`: SmoothGrad on the inputs **plus** multiplicative noise on the **weights** (NoiseGrad), averaged.
* `ContrastiveAttribution`: wraps any method to explain `logit[target] − logit[negative_target]`, or `logit[target] − mean(other logits)`.

Note: on RegNet, grey occlusion patches often *increase* the class score (red), even on the bird itself: occluded images are out of distribution, and this model reacts to them in odd ways. Occlusion maps are cleaner on VGG16 (see the extended notebook): the method's behaviour depends on the model.

In [ ]:
from extra_attributions import TiledOcclusion, FusionGrad, ContrastiveAttribution

K = 4         # change me: resolution factor of TiledOcclusion (must divide WINDOW): 1, 2, 4, 8
WINDOW = 32   # change me: occlusion window, e.g. 16, 32, 64

for t in TARGETS:
    attr_occ = Occlusion(model).attribute(x, target=t, sliding_window_shapes=(3, WINDOW, WINDOW),
                                          strides=(3, WINDOW, WINDOW), baselines=0, perturbations_per_eval=32)
    attr_tocc = TiledOcclusion(model).attribute(x, target=t, k=(1, K, K), window=(3, WINDOW, WINDOW),
                                                baselines=0, perturbations_per_eval=32, show_progress=False)
    plot_attributions(x_img, [attr_occ, attr_tocc], [f'Occlusion (w=s={WINDOW})', f'TiledOcclusion (w={WINDOW}, k={K})'],
                      title=f'Target: {idx_to_label(t)}')

In [ ]:
WEIGHT_NOISE = 0.1   # change me: std of the multiplicative noise on the weights, e.g. 0.05, 0.1, 0.2

# FusionGrad perturbs the weights in place, so we give it a copy of the model
fg_model = copy.deepcopy(model)
fusiongrad = FusionGrad(Saliency(fg_model), model=fg_model)
stdev = 0.15 * (x.max() - x.min()).item()
for t in TARGETS:
    attr_ng = fusiongrad.attribute(x, target=t, std=WEIGHT_NOISE, mean=1., n=8, additive_noise=False,   # weight noise only (NoiseGrad)
                                   sg_std=0., m=1, sg_additive_noise=True, show_progress=False, abs=True)
    attr_fg = fusiongrad.attribute(x, target=t, std=WEIGHT_NOISE, mean=1., n=4, additive_noise=False,   # weight + input noise
                                   sg_std=stdev, m=4, sg_additive_noise=True, show_progress=False, abs=True)
    attr_sg = NoiseTunnel(Saliency(model)).attribute(x, nt_type='smoothgrad', nt_samples=16, nt_samples_batch_size=8,
                                                     stdevs=stdev, target=t, abs=True)
    plot_attributions(x_img, [Saliency(model).attribute(x, target=t, abs=True), attr_sg, attr_ng, attr_fg],
                      ['|Saliency|', 'SmoothGrad', 'NoiseGrad', 'FusionGrad'], title=f'Target: {idx_to_label(t)} (absolute gradients)')

In [ ]:
# ContrastiveAttribution (+ TiledOcclusion): CLASS_A vs. CLASS_B (Section 5), or vs. the average of all other classes
tocc_kw = dict(k=(1, 2, 2), window=(3, 32, 32), baselines=0, perturbations_per_eval=32, show_progress=False)
attr_a = TiledOcclusion(model).attribute(xc, target=a, **tocc_kw)
attr_vs_b = ContrastiveAttribution(TiledOcclusion(model)).attribute(xc, target=a, negative_target=b, **tocc_kw)
attr_vs_all = ContrastiveAttribution(TiledOcclusion(model)).attribute(xc, target=a, negative_target='other_classes', **tocc_kw)
plot_attributions(xc_img, [attr_a, attr_vs_b, attr_vs_all],
                  [f'TiledOcc: {CLASS_A}', f'{CLASS_A} − {CLASS_B}', f'{CLASS_A} − mean(others)'])

## Take-home messages

* Different methods answer **different questions**, and their hyperparameters (baseline, noise, layer...) matter as much as the method.
* **Visually appealing ≠ faithful**: Guided Backprop gives beautiful maps that barely depend on the class or on the model.
* **Contrastive** explanations (*why A rather than B?*) often answer the user's real question better.